# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.91620641 -0.77279409  0.33048932  0.2640768   0.34484474]
 [ 0.5125053  -0.75169093 -0.87127399 -0.21405076 -0.53944607]
 [-0.0537074   0.10669794 -0.50372155 -0.07859156  0.52321436]
 [-0.19836865  0.98750004 -0.8973727  -0.86936949 -0.90326119]
 [-0.1593579  -0.30111161 -0.59061157 -0.08986537  0.21175906]
 [ 0.22240962 -0.1185493  -0.64476461  0.57617029  0.22601114]
 [ 0.86726502 -0.53260971 -0.19891196 -0.02842786 -0.48005051]
 [ 0.53064069  0.0428547   0.53144329  0.48852445 -0.9971394 ]
 [ 0.26625276 -0.17786302  0.79713303  0.77805245 -0.45259756]
 [ 0.7735104  -0.34739189  0.46862325  0.03816798 -0.38249249]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a2', 'a1', 'a2', 'a2', 'a2', 'a2', 'a2', 'a1', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 0, 1, 0, 0, 1, 1, 1, 1, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/25 [00:00<?, ?it/s]

SVI:   4%|▍         | 1/25 [00:01<00:24,  1.01s/it]

SVI:   4%|▍         | 1/25 [00:01<00:24,  1.01s/it, loss=2064.4001]

SVI:   8%|▊         | 2/25 [00:01<00:23,  1.01s/it, loss=2553.7463]

SVI:  12%|█▏        | 3/25 [00:01<00:22,  1.01s/it, loss=2639.9485]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.12it/s, loss=2212.9353]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.12it/s, loss=2397.1221]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.12it/s, loss=3001.5574]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.33it/s, loss=3001.5574]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.33it/s, loss=1813.2456]